# Demo GenMax - Khai phá tập phổ biến tối đại và Sinh Luật kết hợp


## 1. Cài đặt môi trường và Khởi tạo

In [1]:
using Random
Random.seed!(42)

println("Đang import các thư viện từ thư mục src...")
include("../src/structures.jl")
include("../src/algorithm/genmax.jl")
include("../src/utils.jl")

using .Structures
using .GenMaxAlgo
using .Utils

println("Import thành công!")

Đang import các thư viện từ thư mục src...
Import thành công!


## 2. Minh họa trên Dữ liệu Cơ sở (Toy Dataset)
Sử dụng dữ liệu nhỏ để kiểm thử tính đúng đắn của logic cốt lõi. Đầu tiên, chúng ta sẽ tạo file dữ liệu `toy.txt` (nếu chưa có) và hiển thị kết quả sao cho dễ dàng đối chiếu được với cách tính tay/SPMF của Chương 2.

In [2]:
toy_dir = "../data/toy/"
toy_path = joinpath(toy_dir, "toy.txt")

if !isdir(toy_dir)
    mkdir(toy_dir)
end

open(toy_path, "w") do f
    println(f, "1 2 5")
    println(f, "2 4")
    println(f, "2 3")
    println(f, "1 2 4")
    println(f, "1 3")
    println(f, "2 3")
    println(f, "1 3")
    println(f, "1 2 3 5")
    println(f, "1 2 3")
end

println("Đã lưu Toy Dataset: $toy_path")

Đã lưu Toy Dataset: ../data/toy/toy.txt


In [3]:
# Đọc dữ liệu từ file vừa tạo
toy_data = read_spmf_file(toy_path)
println("\n* Dữ liệu các giao dịch (Transactions):")
for (i, t) in enumerate(toy_data)
    println("  TID $i : ", Base.join(t, " "))
end

minsup_toy = 2
println("\n* Chạy GenMax với minsup = $minsup_toy...")
mfi_toy = genmax(toy_data, minsup_toy)

println("\n* Các tập phổ biến tối đại (Maximal Frequent Itemsets):")
for (itemset, sup) in mfi_toy
    println("  Itemset: ", Base.join(itemset, " "), " | Support: ", sup)
end


* Dữ liệu các giao dịch (Transactions):
  TID 1 : 1 2 5
  TID 2 : 2 4
  TID 3 : 2 3
  TID 4 : 1 2 4
  TID 5 : 1 3
  TID 6 : 2 3
  TID 7 : 1 3
  TID 8 : 1 2 3 5
  TID 9 : 1 2 3

* Chạy GenMax với minsup = 2...

* Các tập phổ biến tối đại (Maximal Frequent Itemsets):
  Itemset: 2 4 | Support: 2
  Itemset: 1 2 5 | Support: 2
  Itemset: 1 2 3 | Support: 2


## 3. Chạy thử nghiệm trên Dữ liệu Benchmark
Ở bước này, chúng ta sẽ áp dụng thuật toán GenMax với tập dữ liệu benchmark thực tế để chứng minh code hoạt động trơn tru không có lỗi về quản lý I/O, bộ nhớ hay logic trên dữ liệu lớn.
Chúng ta sẽ thử với một dataset quen thuộc: `chess.dat`.

In [4]:
benchmark_path = "../data/benchmark/chess.dat"
println("Đang đọc dữ liệu: ", benchmark_path)
chess_data = read_spmf_file(benchmark_path)
println("Đã đọc ", length(chess_data), " giao dịch.")

minsup_chess = 0.8  # 80%
println("\nKhởi chạy GenMax trên Chess dataset với minsup =", minsup_chess, "...")

# Sử dụng @time để đo đạc thời gian và memory
mfi_chess = @time genmax(chess_data, minsup_chess)

println("\nHoàn tất! Số lượng Maximal Frequent Itemsets tìm được: ", length(mfi_chess))
println("Top 5 MFI đầu tiên:")
for i in 1:min(5, length(mfi_chess))
    println("  ", Base.join(mfi_chess[i][1], " "), " (Support = ", mfi_chess[i][2], ")")
end

Đang đọc dữ liệu: ../data/benchmark/chess.dat
Đã đọc 3196 giao dịch.

Khởi chạy GenMax trên Chess dataset với minsup =0.8...
  0.089027 seconds (99.78 k allocations: 6.820 MiB, 91.30% compilation time)

Hoàn tất! Số lượng Maximal Frequent Itemsets tìm được: 228
Top 5 MFI đầu tiên:
  46 (Support = 2556)
  44 58 60 (Support = 2564)
  29 40 44 52 58 (Support = 2559)
  29 40 52 58 60 64 (Support = 2569)
  29 42 5 (Support = 2556)


## 4. Demo Ứng dụng thực tế: Sinh Luật Kết Hợp (Association Rules)
Sau khi trích xuất được MFI, quá trình Market Basket Analysis (Phân tích Giỏ hàng) thường được tiếp nối bằng việc sinh luật. 

Hàm dưới đây sẽ tính toán và sinh luật kết hợp dạng `A -> B` từ các Frequent Itemsets trên tập `toy_data`, lọc ra các luật với điều kiện `minconf`, tính giá trị *Lift* (độ tương quan khách quan), sau đó in ra Top 10 luật chất lượng và mạnh nhất dựa trên độ Lift đó.

In [5]:
"""
Hàm quét tham chiếu cơ sở dữ liệu để đếm exact support của 1 sub-itemset.
"""
function get_support(itemset, db)
    count = 0
    for tx in db
        if issubset(itemset, tx)
            count += 1
        end
    end
    return count
end

"""
Sinh luật kết hợp (Association Rules) lấy thông tin từ các Maximal Frequent Itemsets.
"""
function generate_rules(data, mfi_results, minconf)
    rules = []
    total_tx = length(data)
    
    for (mfi, sup_val) in mfi_results
        if length(mfi) < 2
            continue
        end
        
        # Vì là tập Maximal nên toàn bộ tập được sinh đảm bảo lớn hơn minsup
        supp_itemset = sup_val
        
        # Sinh các luật đơn giản A -> B khi biết tập A U B (MFI)
        for i in 1:length(mfi)
            consequent = [mfi[i]]
            antecedent = setdiff(mfi, consequent)
            
            supp_A = get_support(antecedent, data)
            supp_B = get_support(consequent, data)
            
            conf = supp_itemset / supp_A
            if conf >= minconf
                # Tính hệ số Lift đo độ tương quan sự kiện rời rạc
                lift = (supp_itemset / total_tx) / ((supp_A / total_tx) * (supp_B / total_tx))
                push!(rules, (antecedent, consequent, conf, lift))
            end
        end
    end
    
    # Sort các rules dựa vào Lift giảm dần
    sort!(rules, by = x -> x[4], rev = true)
    return rules
end

generate_rules

In [6]:
minconf_val = 0.7
println("Bắt đầu quy trình kiểm thử sinh luật kết hợp trên dữ liệu Toy Dataset...")
println("Điều kiện đánh giá: Minsup = ", minsup_toy, " | Minconf = ", minconf_val, "\n")

rules = generate_rules(toy_data, mfi_toy, minconf_val)

println("--- TOP 10 LUẬT KẾT HỢP (THEO ĐỘ MẠNH 'LIFT') ---")
show_limit = min(10, length(rules))
for i in 1:show_limit
    ant, con, conf, lift_val = rules[i]
    ant_str = Base.join(ant, ", ")
    con_str = Base.join(con, ", ")
    
    println("Rule ", rpad(i, 2), ": {", rpad(ant_str, 5), "} -> {", rpad(con_str, 3), "} | Conf: ", round(conf, digits=2), " | Lift: ", round(lift_val, digits=2))
end

if length(rules) == 0
    println("Không tìm thấy quy tắc nào thỏa điều kiện đặt ra!")
end

Bắt đầu quy trình kiểm thử sinh luật kết hợp trên dữ liệu Toy Dataset...
Điều kiện đánh giá: Minsup = 2 | Minconf = 0.7

--- TOP 10 LUẬT KẾT HỢP (THEO ĐỘ MẠNH 'LIFT') ---
Rule 1 : {2, 5 } -> {1  } | Conf: 1.0 | Lift: 1.5
Rule 2 : {4    } -> {2  } | Conf: 1.0 | Lift: 1.29
Rule 3 : {1, 5 } -> {2  } | Conf: 1.0 | Lift: 1.29
